# Parameter-shift gradients

Differentiate the same QNode with the hardware-compatible parameter-shift rule on both devices.

The SDK reference and MettleQ calls below use the same circuit and result contract. Timing includes the complete call shown.

In [ ]:
import numpy as np
import pennylane as qml
from pennylane import numpy as pnp

from mettleq.integrations.pennylane import MettleQDevice
from tutorials._support import (
    benchmark,
    emit_result,
    max_abs_error,
    pennylane_selection,
    phase_aligned_statevector_error,
    total_variation_distance,
)

In [ ]:
def make_qnode(device):
    @qml.qnode(device, diff_method="parameter-shift")
    def circuit(theta):
        qml.RX(theta, wires=0)
        qml.RY(-0.23, wires=1)
        qml.CNOT(wires=[0, 1])
        return qml.expval(qml.Z(1))
    return circuit

theta = pnp.array(0.41, requires_grad=True)
reference_qnode = make_qnode(qml.device("default.qubit", wires=2))
def reference_value_gradient():
    return float(reference_qnode(theta)), float(qml.grad(reference_qnode)(theta))
reference, reference_ms, _ = benchmark(reference_value_gradient)
mettleq_device = MettleQDevice(wires=2, method="statevector", device="cpu")
mettleq_qnode = make_qnode(mettleq_device)
def mettleq_value_gradient():
    return float(mettleq_qnode(theta)), float(qml.grad(mettleq_qnode)(theta))
candidate, mettleq_ms, _ = benchmark(mettleq_value_gradient)
error = max_abs_error(reference, candidate)
method, device = pennylane_selection(mettleq_device)
tutorial_result = emit_result(
    notebook="pennylane/03_parameter_shift_gradients.ipynb",
    framework="pennylane",
    reference_ms=reference_ms,
    mettleq_ms=mettleq_ms,
    check="value and parameter-shift gradient atol=3e-6",
    passed=error <= 3e-6,
    exact_match=reference == candidate,
    selected_method=method,
    selected_device=device,
    metrics={"max_value_or_gradient_error": error, "reference": reference, "mettleq": candidate},
)